# 06 - Model, Data, Infrastructure Monitoring

This notebook satisfies the Week 5 project steps:

1. Implement model monitors on the ML system.
2. Implement data monitors on the ML system.
3. Implement infrastructure monitors on the ML system.
4. Create a CloudWatch monitoring dashboard for the SageMaker batch job.
5. Generate model and data reports for SageMaker deliverables.

The project uses SageMaker Batch Transform as the first deployment target, so the monitoring workflow focuses on batch output quality, data drift, and SageMaker training/transform job health rather than a long-running endpoint.


In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sagemaker
import seaborn as sns

from plot_style import PURPLE_DARK, PURPLE_LIGHT, PURPLE_PRIMARY, apply_style

apply_style()
pd.set_option("display.max_colwidth", 120)


## 1. Configure AWS and project paths

The notebook uses the same S3 bucket and split layout from the previous modules. It also reads the Week 4 model summary to locate the model artifact and Batch Transform output.


In [ ]:
%store -r bucket
%store -r role
%store -r region

boto_session = boto3.Session()
sts = boto3.client("sts")
identity = sts.get_caller_identity()
account_id = identity["Account"]
region = globals().get("region") or boto_session.region_name

if globals().get("role"):
    role = globals()["role"]
elif hasattr(sagemaker, "get_execution_role"):
    role = sagemaker.get_execution_role()
else:
    # SageMaker Studio/Learner Lab often runs as an assumed role. Convert that
    # STS ARN back to the IAM role ARN expected by SageMaker APIs.
    caller_arn = identity["Arn"]
    if ":assumed-role/" in caller_arn:
        role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
        role = f"arn:aws:iam::{account_id}:role/{role_name}"
    else:
        role = caller_arn

bucket = globals().get("bucket") or f"yelp-sentiment-mlops-{account_id}"

s3 = boto3.client("s3")
sagemaker_client = boto3.client("sagemaker")
cloudwatch = boto3.client("cloudwatch")

split_prefix = "processed/splits"
monitoring_prefix = "monitoring/week5"
cloudwatch_namespace = "YelpSentimentMLOps/Monitoring"
dashboard_name = "yelp-sentiment-week5-monitoring"

report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)

train_s3 = f"s3://{bucket}/{split_prefix}/train/train.csv"
validation_s3 = f"s3://{bucket}/{split_prefix}/validation/validation.csv"
test_s3 = f"s3://{bucket}/{split_prefix}/test/test.csv"
production_s3 = f"s3://{bucket}/{split_prefix}/production/production.csv"

model_summary_path = report_dir / "model_evaluation_summary.md"
metrics_path = report_dir / "benchmark_vs_model_metrics.csv"

print("Region:", region)
print("Bucket:", bucket)
print("CloudWatch namespace:", cloudwatch_namespace)


## 2. Load split data and Week 4 model metrics

These inputs become the baseline for model, data, and infrastructure monitoring.


In [ ]:
train_df = pd.read_csv(train_s3)
validation_df = pd.read_csv(validation_s3)
test_df = pd.read_csv(test_s3)
production_df = pd.read_csv(production_s3)
metrics_df = pd.read_csv(metrics_path)

print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))
print("Production rows:", len(production_df))
display(metrics_df)


## 3. Parse Week 4 artifact locations

The Week 4 summary contains the trained model artifact and Batch Transform output S3 path. We parse those paths so the monitoring notebook can inspect the latest deployed batch workflow.


In [ ]:
summary_text = model_summary_path.read_text(encoding="utf-8")

def extract_backtick_value(label, text):
    pattern = rf"- {re.escape(label)}: `([^`]+)`"
    match = re.search(pattern, text)
    return match.group(1) if match else None

model_artifact_s3 = extract_backtick_value("Model artifact", summary_text)
batch_input_s3 = extract_backtick_value("Batch input", summary_text)
batch_output_s3 = extract_backtick_value("Batch output", summary_text)

training_job_name = None
if model_artifact_s3:
    match = re.search(r"/(sagemaker-scikit-learn-[^/]+)/output/model.tar.gz$", model_artifact_s3)
    training_job_name = match.group(1) if match else None

print("Model artifact:", model_artifact_s3)
print("Training job:", training_job_name)
print("Batch input:", batch_input_s3)
print("Batch output:", batch_output_s3)


## 4. Data monitors

The data monitor compares baseline training data against the reserved production split. It checks missing text, review length, label distribution, and basic drift thresholds.


In [ ]:
def split_quality_row(split_name, df):
    text = df["clean_text"].fillna("").astype(str)
    word_counts = text.str.split().map(len)
    row = {
        "split": split_name,
        "row_count": int(len(df)),
        "missing_clean_text_count": int((text.str.len() == 0).sum()),
        "missing_clean_text_rate": float((text.str.len() == 0).mean()),
        "avg_review_word_count": float(word_counts.mean()),
        "p95_review_word_count": float(word_counts.quantile(0.95)),
    }
    if "sentiment_label" in df.columns:
        row["positive_share"] = float(df["sentiment_label"].astype(int).mean())
    return row

data_quality_df = pd.DataFrame(
    [
        split_quality_row("train", train_df),
        split_quality_row("validation", validation_df),
        split_quality_row("test", test_df),
        split_quality_row("production", production_df),
    ]
)

train_baseline = data_quality_df[data_quality_df["split"] == "train"].iloc[0]
production_row = data_quality_df[data_quality_df["split"] == "production"].iloc[0]

data_monitor_summary = {
    "missing_text_rate": production_row["missing_clean_text_rate"],
    "missing_text_threshold": 0.01,
    "missing_text_status": "PASS" if production_row["missing_clean_text_rate"] <= 0.01 else "ALERT",
    "avg_word_count_baseline": train_baseline["avg_review_word_count"],
    "avg_word_count_production": production_row["avg_review_word_count"],
    "avg_word_count_relative_drift": float(
        abs(production_row["avg_review_word_count"] - train_baseline["avg_review_word_count"])
        / max(train_baseline["avg_review_word_count"], 1)
    ),
    "avg_word_count_relative_drift_threshold": 0.20,
    "positive_share_baseline": train_baseline.get("positive_share"),
    "positive_share_production": production_row.get("positive_share"),
    "positive_share_abs_drift": float(abs(production_row.get("positive_share", 0) - train_baseline.get("positive_share", 0))),
    "positive_share_abs_drift_threshold": 0.15,
}
data_monitor_summary["word_count_drift_status"] = (
    "PASS" if data_monitor_summary["avg_word_count_relative_drift"] <= data_monitor_summary["avg_word_count_relative_drift_threshold"] else "ALERT"
)
data_monitor_summary["label_drift_status"] = (
    "PASS" if data_monitor_summary["positive_share_abs_drift"] <= data_monitor_summary["positive_share_abs_drift_threshold"] else "ALERT"
)

data_quality_path = report_dir / "data_monitoring_metrics.csv"
data_summary_path = report_dir / "data_monitoring_summary.json"
data_quality_df.to_csv(data_quality_path, index=False)
data_summary_path.write_text(json.dumps(data_monitor_summary, indent=2), encoding="utf-8")

display(data_quality_df)
print("Wrote", data_quality_path)
print("Wrote", data_summary_path)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(
    data=data_quality_df,
    x="split",
    y="avg_review_word_count",
    hue="split",
    palette=[PURPLE_DARK, PURPLE_PRIMARY, "#9e9ac8", PURPLE_LIGHT],
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Average Review Word Count")
axes[0].set_xlabel("Split")
axes[0].set_ylabel("Words")

sns.barplot(
    data=data_quality_df,
    x="split",
    y="positive_share",
    hue="split",
    palette=[PURPLE_DARK, PURPLE_PRIMARY, "#9e9ac8", PURPLE_LIGHT],
    legend=False,
    ax=axes[1],
)
axes[1].axhline(train_baseline["positive_share"], color="#444444", linestyle="--", linewidth=1)
axes[1].set_ylim(0, 1)
axes[1].set_title("Positive Share by Split")
axes[1].set_xlabel("Split")
axes[1].set_ylabel("Positive share")

plt.tight_layout()
data_plot_path = report_dir / "data_monitoring_dashboard.png"
plt.savefig(data_plot_path, dpi=150)
plt.show()
print("Wrote", data_plot_path)


## 5. Model monitors

The model monitor checks test-set model quality and prediction distribution from the Batch Transform output. Macro F1 is the quality gate metric from the design document.


In [ ]:
trained_test = metrics_df[(metrics_df["model"] == "tfidf_logistic_regression") & (metrics_df["split"] == "test")].iloc[0]
benchmark_test = metrics_df[(metrics_df["model"] == "majority_class_benchmark") & (metrics_df["split"] == "test")].iloc[0]

def parse_s3_uri(s3_uri):
    bucket_name, key = s3_uri.replace("s3://", "", 1).split("/", 1)
    return bucket_name, key

def load_batch_predictions(output_s3_uri):
    if not output_s3_uri:
        return pd.DataFrame()
    output_bucket, output_prefix = parse_s3_uri(output_s3_uri)
    response = s3.list_objects_v2(Bucket=output_bucket, Prefix=output_prefix)
    records = []
    for obj in response.get("Contents", []):
        key = obj["Key"]
        if key.endswith("/") or key.endswith(".failure"):
            continue
        body = s3.get_object(Bucket=output_bucket, Key=key)["Body"].read().decode("utf-8")
        for line in body.splitlines():
            if not line.strip():
                continue
            parsed = json.loads(line)
            if isinstance(parsed, list):
                records.extend(parsed)
            elif isinstance(parsed, dict):
                records.append(parsed)
    return pd.DataFrame(records)

predictions_df = load_batch_predictions(batch_output_s3)
if predictions_df.empty:
    print("No Batch Transform prediction output found yet. Re-run notebook 05 Batch Transform if this is unexpected.")
else:
    display(predictions_df.head())

prediction_positive_share = None
prediction_count = 0
prediction_drift = None
if not predictions_df.empty and "predicted_label" in predictions_df.columns:
    prediction_count = int(len(predictions_df))
    prediction_positive_share = float(predictions_df["predicted_label"].astype(int).mean())
    prediction_drift = float(abs(prediction_positive_share - test_df["sentiment_label"].astype(int).mean()))

model_monitor_summary = {
    "quality_gate_metric": "macro_f1",
    "quality_gate_threshold": 0.80,
    "benchmark_test_macro_f1": float(benchmark_test["f1_macro"]),
    "trained_model_test_macro_f1": float(trained_test["f1_macro"]),
    "trained_model_test_accuracy": float(trained_test["accuracy"]),
    "quality_gate_status": "PASS" if float(trained_test["f1_macro"]) >= 0.80 else "ALERT",
    "batch_prediction_count": prediction_count,
    "batch_prediction_positive_share": prediction_positive_share,
    "prediction_positive_share_drift_vs_test": prediction_drift,
    "prediction_positive_share_drift_threshold": 0.15,
    "prediction_distribution_status": "PASS" if prediction_drift is None or prediction_drift <= 0.15 else "ALERT",
}

model_summary_path = report_dir / "model_monitoring_summary.json"
model_summary_path.write_text(json.dumps(model_monitor_summary, indent=2), encoding="utf-8")
print(json.dumps(model_monitor_summary, indent=2))
print("Wrote", model_summary_path)


In [ ]:
model_plot_df = metrics_df.copy()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=model_plot_df, x="split", y="f1_macro", hue="model", palette=[PURPLE_LIGHT, PURPLE_DARK], ax=ax)
ax.axhline(0.80, color="#444444", linestyle="--", linewidth=1, label="quality gate 0.80")
ax.set_ylim(0, 1)
ax.set_title("Benchmark vs Trained Model Macro F1")
ax.set_xlabel("Split")
ax.set_ylabel("Macro F1")
ax.legend(loc="lower right")
plt.tight_layout()
model_plot_path = report_dir / "model_monitoring_dashboard.png"
plt.savefig(model_plot_path, dpi=150)
plt.show()
print("Wrote", model_plot_path)


## 6. Bias and explainability monitors

The lecture calls out bias and explainability monitoring as part of model monitoring. For this TF-IDF Logistic Regression model, we add two lightweight monitors:

- **Bias monitor:** compares class balance before the model (`sentiment_label`) and after the model (`predicted_label`) to detect output imbalance.
- **Explainability proxy:** tracks drift in high-frequency review terms between training and production data. This is not full SHAP/Clarify explainability, but it gives a practical signal for whether the model is seeing different language patterns than it was trained on.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bias_rows = []
for split_name, df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
    "production_input": production_df,
}.items():
    bias_rows.append(
        {
            "source": split_name,
            "stage": "pre_model",
            "row_count": int(len(df)),
            "positive_share": float(df["sentiment_label"].astype(int).mean()),
            "negative_share": float(1 - df["sentiment_label"].astype(int).mean()),
        }
    )

if not predictions_df.empty and "predicted_label" in predictions_df.columns:
    predicted_positive_share = float(predictions_df["predicted_label"].astype(int).mean())
    bias_rows.append(
        {
            "source": "batch_transform_output",
            "stage": "post_model",
            "row_count": int(len(predictions_df)),
            "positive_share": predicted_positive_share,
            "negative_share": float(1 - predicted_positive_share),
        }
    )
else:
    predicted_positive_share = None

bias_monitor_df = pd.DataFrame(bias_rows)
baseline_positive_share = float(train_df["sentiment_label"].astype(int).mean())
post_model_bias_drift = (
    abs(predicted_positive_share - baseline_positive_share) if predicted_positive_share is not None else None
)

vectorizer = CountVectorizer(max_features=1000, stop_words="english", min_df=5)
train_counts = vectorizer.fit_transform(train_df["clean_text"].fillna("").astype(str))
production_counts = vectorizer.transform(production_df["clean_text"].fillna("").astype(str))
terms = np.array(vectorizer.get_feature_names_out())

train_rates = np.asarray(train_counts.sum(axis=0)).ravel()
production_rates = np.asarray(production_counts.sum(axis=0)).ravel()
train_rates = train_rates / max(train_rates.sum(), 1)
production_rates = production_rates / max(production_rates.sum(), 1)
term_drift = np.abs(production_rates - train_rates)
top_drift_idx = np.argsort(term_drift)[-20:][::-1]

explainability_drift_df = pd.DataFrame(
    {
        "term": terms[top_drift_idx],
        "train_frequency_share": train_rates[top_drift_idx],
        "production_frequency_share": production_rates[top_drift_idx],
        "absolute_drift": term_drift[top_drift_idx],
    }
)

explainability_proxy_summary = {
    "method": "TF-IDF explainability proxy using token frequency drift",
    "note": "For a future transformer or endpoint version, SageMaker Clarify/SHAP can replace this proxy.",
    "max_term_frequency_drift": float(explainability_drift_df["absolute_drift"].max()),
    "mean_top20_term_frequency_drift": float(explainability_drift_df["absolute_drift"].mean()),
    "top_drift_terms": explainability_drift_df["term"].head(10).tolist(),
}

bias_monitor_summary = {
    "baseline_positive_share": baseline_positive_share,
    "post_model_positive_share": predicted_positive_share,
    "post_model_positive_share_drift": post_model_bias_drift,
    "post_model_positive_share_drift_threshold": 0.15,
    "post_model_bias_status": "PASS" if post_model_bias_drift is None or post_model_bias_drift <= 0.15 else "ALERT",
    "pre_model_bias_note": "Production input class balance is monitored using sentiment_label because this is a simulated production split for the class project.",
    "post_model_bias_note": "Batch Transform predicted class balance is monitored when prediction output is available.",
}

bias_path = report_dir / "bias_monitoring_metrics.csv"
explainability_path = report_dir / "explainability_drift_metrics.csv"
bias_summary_path = report_dir / "bias_explainability_monitoring_summary.json"
bias_monitor_df.to_csv(bias_path, index=False)
explainability_drift_df.to_csv(explainability_path, index=False)
bias_summary_path.write_text(
    json.dumps(
        {
            "bias_monitor": bias_monitor_summary,
            "explainability_proxy_monitor": explainability_proxy_summary,
        },
        indent=2,
    ),
    encoding="utf-8",
)

display(bias_monitor_df)
display(explainability_drift_df.head(10))
print("Wrote", bias_path)
print("Wrote", explainability_path)
print("Wrote", bias_summary_path)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(
    data=bias_monitor_df,
    x="source",
    y="positive_share",
    hue="stage",
    palette=[PURPLE_LIGHT, PURPLE_DARK],
    ax=axes[0],
)
axes[0].axhline(baseline_positive_share, color="#444444", linestyle="--", linewidth=1)
axes[0].set_ylim(0, 1)
axes[0].set_title("Bias Monitor: Positive Share")
axes[0].set_xlabel("Source")
axes[0].set_ylabel("Positive share")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(
    data=explainability_drift_df.head(10),
    x="absolute_drift",
    y="term",
    color=PURPLE_PRIMARY,
    ax=axes[1],
)
axes[1].set_title("Explainability Proxy: Top Term Drift")
axes[1].set_xlabel("Absolute frequency drift")
axes[1].set_ylabel("Term")

plt.tight_layout()
bias_explainability_plot_path = report_dir / "bias_explainability_monitoring_dashboard.png"
plt.savefig(bias_explainability_plot_path, dpi=150)
plt.show()
print("Wrote", bias_explainability_plot_path)


## 7. Failure modes and monitoring practices

Monitoring is useful only when alerts map back to likely failure causes. This section documents the traditional software failures, ML-specific failures, and best practices covered in the lecture so the final project explains why each monitor exists.


In [ ]:
failure_modes = [
    {
        "category": "traditional_software",
        "failure_mode": "logic_error",
        "project_monitor": "Notebook/script validation, report sanity checks, model quality metrics",
        "mitigation": "Review failed cells/logs and rerun validation before deployment.",
    },
    {
        "category": "traditional_software",
        "failure_mode": "integration_or_deployment_error",
        "project_monitor": "SageMaker training job status and Batch Transform job status",
        "mitigation": "CloudWatch alarm if training or transform job does not complete.",
    },
    {
        "category": "traditional_software",
        "failure_mode": "dependency_change",
        "project_monitor": "requirements.txt plus SageMaker training logs",
        "mitigation": "Pin compatible SageMaker/scikit-learn versions for reproducible jobs.",
    },
    {
        "category": "traditional_software",
        "failure_mode": "hardware_or_downtime",
        "project_monitor": "SageMaker job status and CloudWatch infrastructure metrics",
        "mitigation": "Alert on failed jobs and inspect CloudWatch logs/metrics.",
    },
    {
        "category": "ml_specific",
        "failure_mode": "data_distribution_shift",
        "project_monitor": "Review length drift, class balance drift, term frequency drift",
        "mitigation": "Investigate input changes and retrain if drift persists.",
    },
    {
        "category": "ml_specific",
        "failure_mode": "edge_cases",
        "project_monitor": "Missing/empty text rate and outlier review length checks",
        "mitigation": "Add preprocessing guards and collect examples for retraining.",
    },
    {
        "category": "ml_specific",
        "failure_mode": "degenerate_feedback_loop",
        "project_monitor": "Prediction distribution and business proxy metrics over time",
        "mitigation": "Review whether predictions influence future review sampling or business decisions.",
    },
]

best_practices = [
    "Start with a small number of critical metrics: macro F1, missing text rate, class drift, and job success.",
    "Use CloudWatch dashboards and alarms so failures are visible without reading notebook outputs.",
    "Include context in reports: model artifact path, batch output path, thresholds, and status.",
    "Review thresholds regularly to reduce alert fatigue.",
    "Protect monitoring artifacts because review text and identifiers may contain sensitive information.",
    "Collaborate with the team to decide which business metric should be monitored in the final version.",
]

failure_modes_path = report_dir / "monitoring_failure_modes.md"
lines = [
    "# Monitoring Failure Modes and Best Practices",
    "",
    "## Failure Modes",
    "",
    "| Category | Failure Mode | Project Monitor | Mitigation |",
    "|---|---|---|---|",
]
for item in failure_modes:
    lines.append(
        f"| {item['category']} | {item['failure_mode']} | {item['project_monitor']} | {item['mitigation']} |"
    )
lines.extend(["", "## Monitoring Best Practices", ""])
lines.extend([f"- {practice}" for practice in best_practices])

failure_modes_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Wrote", failure_modes_path)
print("\n".join(lines))


## 8. Infrastructure monitors

For the batch workflow, infrastructure monitoring checks the latest SageMaker training job and latest Batch Transform job status. It publishes simple success/failure custom metrics to CloudWatch.


In [ ]:
training_status = "UNKNOWN"
training_secondary_status = None
training_billable_seconds = None
if training_job_name:
    training_description = sagemaker_client.describe_training_job(TrainingJobName=training_job_name)
    training_status = training_description.get("TrainingJobStatus", "UNKNOWN")
    training_secondary_status = training_description.get("SecondaryStatus")
    training_billable_seconds = training_description.get("BillableTimeInSeconds")
else:
    training_description = {}

transform_jobs = sagemaker_client.list_transform_jobs(SortBy="CreationTime", SortOrder="Descending", MaxResults=10).get("TransformJobSummaries", [])
latest_transform_job = transform_jobs[0] if transform_jobs else None
transform_job_name = latest_transform_job["TransformJobName"] if latest_transform_job else None
transform_status = latest_transform_job.get("TransformJobStatus", "UNKNOWN") if latest_transform_job else "UNKNOWN"
transform_description = (
    sagemaker_client.describe_transform_job(TransformJobName=transform_job_name) if transform_job_name else {}
)

infrastructure_summary = {
    "training_job_name": training_job_name,
    "training_job_status": training_status,
    "training_secondary_status": training_secondary_status,
    "training_billable_seconds": training_billable_seconds,
    "training_job_succeeded": 1 if training_status == "Completed" else 0,
    "latest_transform_job_name": transform_job_name,
    "latest_transform_job_status": transform_status,
    "latest_transform_job_succeeded": 1 if transform_status == "Completed" else 0,
    "latest_transform_output_path": transform_description.get("TransformOutput", {}).get("S3OutputPath"),
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
}

infrastructure_path = report_dir / "infrastructure_monitoring_summary.json"
infrastructure_path.write_text(json.dumps(infrastructure_summary, indent=2, default=str), encoding="utf-8")
print(json.dumps(infrastructure_summary, indent=2, default=str))
print("Wrote", infrastructure_path)


## 9. Publish monitoring metrics and alarms to CloudWatch

These custom metrics make the monitoring dashboard independent of whether the project is running a Batch Transform job or a real-time endpoint.


In [ ]:
metric_dimensions = [{"Name": "Project", "Value": "YelpSentimentMLOps"}]
now = datetime.now(timezone.utc)

metric_data = [
    {
        "MetricName": "ModelMacroF1",
        "Dimensions": metric_dimensions,
        "Timestamp": now,
        "Value": model_monitor_summary["trained_model_test_macro_f1"],
        "Unit": "None",
    },
    {
        "MetricName": "DataMissingTextRate",
        "Dimensions": metric_dimensions,
        "Timestamp": now,
        "Value": data_monitor_summary["missing_text_rate"],
        "Unit": "None",
    },
    {
        "MetricName": "DataAvgWordCountDrift",
        "Dimensions": metric_dimensions,
        "Timestamp": now,
        "Value": data_monitor_summary["avg_word_count_relative_drift"],
        "Unit": "None",
    },
    {
        "MetricName": "PredictionPositiveShareDrift",
        "Dimensions": metric_dimensions,
        "Timestamp": now,
        "Value": prediction_drift if prediction_drift is not None else 0.0,
        "Unit": "None",
    },
    {
        "MetricName": "TrainingJobSucceeded",
        "Dimensions": metric_dimensions,
        "Timestamp": now,
        "Value": infrastructure_summary["training_job_succeeded"],
        "Unit": "Count",
    },
    {
        "MetricName": "BatchTransformJobSucceeded",
        "Dimensions": metric_dimensions,
        "Timestamp": now,
        "Value": infrastructure_summary["latest_transform_job_succeeded"],
        "Unit": "Count",
    },
]

cloudwatch.put_metric_data(Namespace=cloudwatch_namespace, MetricData=metric_data)
print(f"Published {len(metric_data)} CloudWatch metrics to {cloudwatch_namespace}")

alarm_specs = [
    ("YelpSentiment-ModelMacroF1-Low", "ModelMacroF1", "LessThanThreshold", 0.80),
    ("YelpSentiment-DataMissingTextRate-High", "DataMissingTextRate", "GreaterThanThreshold", 0.01),
    ("YelpSentiment-WordCountDrift-High", "DataAvgWordCountDrift", "GreaterThanThreshold", 0.20),
    ("YelpSentiment-PredictionDrift-High", "PredictionPositiveShareDrift", "GreaterThanThreshold", 0.15),
    ("YelpSentiment-TrainingJob-Failed", "TrainingJobSucceeded", "LessThanThreshold", 1),
    ("YelpSentiment-BatchTransform-Failed", "BatchTransformJobSucceeded", "LessThanThreshold", 1),
]

for alarm_name, metric_name, comparison, threshold in alarm_specs:
    cloudwatch.put_metric_alarm(
        AlarmName=alarm_name,
        AlarmDescription=f"Week 5 monitor for {metric_name}",
        Namespace=cloudwatch_namespace,
        MetricName=metric_name,
        Dimensions=metric_dimensions,
        Statistic="Average",
        Period=300,
        EvaluationPeriods=1,
        DatapointsToAlarm=1,
        Threshold=threshold,
        ComparisonOperator=comparison,
        TreatMissingData="notBreaching",
    )

print("Created/updated CloudWatch alarms:")
for alarm_name, _, _, _ in alarm_specs:
    print("-", alarm_name)


## 10. Create CloudWatch dashboard

The dashboard contains model, data, and infrastructure monitor widgets from the custom metrics above.


In [ ]:
dashboard_body = {
    "widgets": [
        {
            "type": "text",
            "x": 0,
            "y": 0,
            "width": 24,
            "height": 3,
            "properties": {
                "markdown": "# Yelp Sentiment MLOps Week 5 Monitoring\nModel quality, data quality/drift, and SageMaker batch workflow health."
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 3,
            "width": 12,
            "height": 6,
            "properties": {
                "region": region,
                "title": "Model Monitor: Macro F1",
                "view": "timeSeries",
                "metrics": [[cloudwatch_namespace, "ModelMacroF1", "Project", "YelpSentimentMLOps"]],
                "annotations": {"horizontal": [{"label": "quality gate", "value": 0.80}]},
                "yAxis": {"left": {"min": 0, "max": 1}},
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 3,
            "width": 12,
            "height": 6,
            "properties": {
                "region": region,
                "title": "Data Monitors: Missing Text and Drift",
                "view": "timeSeries",
                "metrics": [
                    [cloudwatch_namespace, "DataMissingTextRate", "Project", "YelpSentimentMLOps"],
                    [".", "DataAvgWordCountDrift", ".", "."],
                    [".", "PredictionPositiveShareDrift", ".", "."],
                ],
                "yAxis": {"left": {"min": 0}},
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 9,
            "width": 12,
            "height": 6,
            "properties": {
                "region": region,
                "title": "Infrastructure Monitor: SageMaker Job Health",
                "view": "timeSeries",
                "metrics": [
                    [cloudwatch_namespace, "TrainingJobSucceeded", "Project", "YelpSentimentMLOps"],
                    [".", "BatchTransformJobSucceeded", ".", "."],
                ],
                "yAxis": {"left": {"min": 0, "max": 1}},
            },
        },
        {
            "type": "text",
            "x": 12,
            "y": 9,
            "width": 12,
            "height": 6,
            "properties": {
                "markdown": f"## Latest SageMaker Jobs\nTraining job: `{training_job_name}`\n\nTransform job: `{transform_job_name}`\n\nBatch output: `{batch_output_s3}`"
            },
        },
    ]
}

dashboard_path = report_dir / "cloudwatch_dashboard_body.json"
dashboard_path.write_text(json.dumps(dashboard_body, indent=2), encoding="utf-8")
cloudwatch.put_dashboard(DashboardName=dashboard_name, DashboardBody=json.dumps(dashboard_body))

dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
print("Wrote", dashboard_path)
print("Created/updated CloudWatch dashboard:", dashboard_name)
print(dashboard_url)


## 11. Generate final Week 5 monitoring report

This markdown report is the main artifact for the tracker and final project documentation.


In [ ]:
report_lines = [
    "# Week 5 Monitoring Report",
    "",
    "## Required Steps Covered",
    "",
    "- Model monitors: macro F1 quality gate, prediction distribution drift, benchmark comparison, bias checks, and explainability proxy drift.",
    "- Data monitors: missing text rate, review length drift, class distribution drift.",
    "- Infrastructure monitors: SageMaker training job status and latest Batch Transform job status.",
    "- CloudWatch dashboard: custom metric dashboard for model, data, and infrastructure monitors.",
    "- SageMaker reports: model, data, infrastructure, bias/explainability, and failure-mode artifacts saved under `reports/`.",
    "",
    "## Model Monitoring Summary",
    "",
    f"- Benchmark test macro F1: {model_monitor_summary['benchmark_test_macro_f1']:.4f}",
    f"- Trained model test macro F1: {model_monitor_summary['trained_model_test_macro_f1']:.4f}",
    f"- Quality gate status: **{model_monitor_summary['quality_gate_status']}**",
    f"- Batch prediction count: {model_monitor_summary['batch_prediction_count']:,}",
    f"- Prediction drift status: **{model_monitor_summary['prediction_distribution_status']}**",
    "",
    "## Data Monitoring Summary",
    "",
    f"- Production missing text rate: {data_monitor_summary['missing_text_rate']:.4f}",
    f"- Missing text status: **{data_monitor_summary['missing_text_status']}**",
    f"- Average word count relative drift: {data_monitor_summary['avg_word_count_relative_drift']:.4f}",
    f"- Word count drift status: **{data_monitor_summary['word_count_drift_status']}**",
    f"- Positive-share drift status: **{data_monitor_summary['label_drift_status']}**",
    "",
    "## Infrastructure Monitoring Summary",
    "",
    f"- Training job: `{training_job_name}` ({training_status})",
    f"- Latest Batch Transform job: `{transform_job_name}` ({transform_status})",
    f"- CloudWatch dashboard: `{dashboard_name}`",
    "",
    "## Artifacts",
    "",
    "- `reports/model_monitoring_summary.json`",
    "- `reports/data_monitoring_metrics.csv`",
    "- `reports/data_monitoring_summary.json`",
    "- `reports/infrastructure_monitoring_summary.json`",
    "- `reports/cloudwatch_dashboard_body.json`",
    "- `reports/model_monitoring_dashboard.png`",
    "- `reports/data_monitoring_dashboard.png`",
]

monitoring_report_path = report_dir / "week5_monitoring_report.md"
monitoring_report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("Wrote", monitoring_report_path)
print("\n".join(report_lines))
